# Interactive Z-table: from the normal PDF to the probability
Move the slider and observe the shaded area, the **integral evaluated numerically**, and the corresponding four-decimal Z-table value.

For $Z\sim N(0,1)$:

$$f_Z(t)=\frac{1}{\sqrt{2\pi}}e^{-u^2/2},\qquad \Phi(z)=P(Z\le z)=\int_{-\infty}^{z}\frac{1}{\sqrt{2\pi}}e^{-u^2/2}\,dt.$$

The integral is evaluated using `scipy.integrate.quad` rather than looking up a precomputed table.

In [4]:
%pip install ipywidgets

In [5]:
# Run once if your Jupyter environment is missing the packages:
# %pip install numpy scipy matplotlib ipywidgets ipympl

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

In [8]:
def standard_normal_pdf(t):
    # Normal PDF
    return np.exp(-np.asarray(t)**2 / 2) / np.sqrt(2 * np.pi)


def normal_cdf_by_integration(z):
    # Numerical integration, not a table lookup
    area, estimated_error = quad(lambda t: float(standard_normal_pdf(t)), -np.inf, z)
    return area, estimated_error


z_slider = widgets.FloatSlider(
    value=1.0, min=-3.5, max=3.5, step=0.05,
    description='Z value:', readout_format='.2f',
    continuous_update=False,
    style={'description_width': '90px'},
    layout=widgets.Layout(width='75%')
)

examples = widgets.Dropdown(
    options=[('Select an example', None), ('Q1(a): z = 2.00', 2.0),
             ('Q1(b): z = -1.00', -1.0), ('Q1(c): z = -0.25', -0.25),
             ('Q1(c): z = 1.00', 1.0), ('Centre: z = 0', 0.0)],
    description='Examples:', style={'description_width': '90px'}
)

plot_output = widgets.Output()


def on_example_change(change):
    if change['new'] is not None:
        z_slider.value = change['new']

examples.observe(on_example_change, names='value')


def update_display(change=None):
    z = z_slider.value
    phi, err = normal_cdf_by_integration(z)
    pdf_height = float(standard_normal_pdf(z))
    area_from_zero, _ = quad(lambda t: float(standard_normal_pdf(t)), 0, abs(z))

    with plot_output:
        clear_output(wait=True)
        x = np.linspace(-4, 4, 1500)
        y = standard_normal_pdf(x)
        fig, ax = plt.subplots(figsize=(10, 4.6))
        ax.plot(x, y, color='navy', lw=2.5, label=r'$f_Z(t)$')
        ax.fill_between(x, 0, y, where=(x <= z), color='cornflowerblue', alpha=0.42)
        ax.axvline(z, color='firebrick', linestyle='--', lw=1.7)
        ax.set(xlim=(-4, 4), ylim=(0, 0.45), xlabel='z (standard deviations)',
               ylabel='Probability density', title='Z-table value = shaded area to the left')
        ax.text(0.02, .95, rf'$z={z:.2f}$' + '\n' + rf'$\Phi(z)={phi:.4f}$',
                va='top', transform=ax.transAxes, fontsize=13,
                bbox=dict(facecolor='white', alpha=.85, edgecolor='lightgray'))
        ax.grid(alpha=0.15)
        plt.show()

        display(Markdown(rf'''### Formula evaluated for $z={z:.2f}$

$$\Phi({z:.2f})=\int_{{-\infty}}^{{{z:.2f}}}\frac{{1}}{{\sqrt{{2\pi}}}}e^{{-u^2/2}}\,dt
\;\approx\; {phi:.8f}$$

**Rounded Z-table entry:** $\boxed{{\Phi({z:.2f})={phi:.4f}}}$  
**Density height at the cutoff:** $f_Z({z:.2f})={pdf_height:.6f}$ (not the probability).  
**Numerical integration error estimate:** {err:.2e}.

Symmetry-based calculation: $\Phi({z:.2f}) = 0.5 {'+' if z >= 0 else '-'} \int_0^{{{abs(z):.2f}}} f_Z(t)\,dt \approx 0.5 {'+' if z >= 0 else '-'} {area_from_zero:.8f} = {phi:.8f}$.
'''))

z_slider.observe(update_display, names='value')
display(widgets.VBox([widgets.HBox([z_slider, examples]), plot_output]))
update_display()

## Verify all three Q1 results by numerical integration

- (a) $P(X<11)=\Phi(2)$
- (b) $P(X>-1)=1-\Phi(-1)$
- (c) $P(2<X<7)=\Phi(1)-\Phi(-0.25)$

In [7]:
phi = lambda z: normal_cdf_by_integration(z)[0]
a = phi(2)
b = 1 - phi(-1)
c = phi(1) - phi(-0.25)
c_table = round(phi(1), 4) - round(phi(-0.25), 4)
print(f'(a) Phi(2.00)                   = {a:.4f}')
print(f'(b) 1 - Phi(-1.00)              = {b:.4f}')
print(f'(c) Direct integral difference = {c:.8f} (rounds to {c:.4f})')
print(f'(c) Using 4-decimal Z-table    = {round(phi(1),4):.4f} - {round(phi(-0.25),4):.4f} = {c_table:.4f}')
print('The tiny discrepancy in (c) is caused by rounding table entries first.')

(a) Phi(2.00)                   = 0.9772
(b) 1 - Phi(-1.00)              = 0.8413
(c) Direct integral difference = 0.44005107 (rounds to 0.4401)
(c) Using 4-decimal Z-table    = 0.8413 - 0.4013 = 0.4400
The tiny discrepancy in (c) is caused by rounding table entries first.
